In this lab, we will build a model to perform segmentation of RNA molecules, seen as surfaces.

![](https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/rna_image.png)

For this, we will first build a network able to process these surfaces, and then train it on the dataset.
We will additionnally try some small variations and check their effects.

# Environment and data

Let's download the dataset and additional necessary packages.

You will need a GPU for this Lab. We recommend to use Google Colab.

In [1]:
!pip install potpourri3d
!pip install git+https://github.com/skoch9/meshplot.git
!pip install pythreejs

!wget https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/RNADataset.zip
!wget https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/material_TD3.zip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 25.7 MB/s  0:00:00
  Cloning https://github.com/skoch9/meshplot.git to /tmp/pip-req-build-qe5xzdf5
  Running command git clone --filter=blob:none --quiet https://github.com/skoch9/meshplot.git /tmp/pip-req-build-qe5xzdf5
  Resolved https://github.com/skoch9/meshplot.git to commit 725e4a7926a5f10888f0edd1762fecf9db751c56
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for meshplot: filename=meshplot-0.3.3-py3-none-any.whl size=21228 sha256=ba70e3aed870398f74573632a530469eca1327652b2d2b99bbfceeb883344708
  Stored in directory: /tmp/pip-ephem-wheel-cache-3pc9lnwm/wheels/05/52/dd/0ab0259df308d57781b9761e43fc9c557771be0cff313e2f8d
Successfully built meshplot
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 21.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 55.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/Geometry processing and Geometric deep learning/MVA_TD4/RNADataset.zip" .
!cp "/content/drive/MyDrive/Geometry processing and Geometric deep learning/MVA_TD4/material_TD3.zip" .


Mounted at /content/drive


In [3]:
!unzip -qq RNADataset
!unzip -qq material_TD3.zip

Code for colab

In [4]:
from google.colab import output
output.enable_custom_widget_manager()

import os

import torch
import torch.nn as nn

import diffusion_utils

# Part I - Implementing Diffusion DiffusionNet

## Goal

In this assignment we will focus on building [DiffusionNet](https://arxiv.org/pdf/2012.00888.pdf), which was presented during the lecture.
This network uses differential geometry tools to compute per-vertex features which can later be used for several tasks such as classification, segmentation or matching.

Recall that the DiffusionNet architechture can be visualized as follows :

![](https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/TD3/DiffusionNet.PNG)


Therefore, following this image, we will build the following objects using PyTorch:
1. **Precomputed operators**. Most of them have been introduced in TD1, the gradient matrix is provided in the `diffusion_utils.py` file provided with this TD (do not modify it!).
2. **Spatial Diffusion**. Using again TD1 (part on spectral diffusion).
3. **Spatial Gradient features**. Using gradient matrices
4. **Per-vertex MLP**. Basic PyTorch
5. **DiffusionNet block** and **DiffusionNet** which consists in assembling the previous 4 parts.

## Reminders on PyTorch
[Pytorch](https://pytorch.org/docs/stable/index.html) is a standard library to build and train neural networks. Two essential components are auto-differentiation and GPU support with parallelization of operations.

A [`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html) object is very similar to a `numpy.array`, and most existing operations in numpy have identical equivalent in PyTorch. However behaviors can be slightly different so always check the documentation ! Using built-in PyTorch functions instead of numpy enables automatic gradient computation.

Note that PyTorch operations are built to be significantly accelerated on GPU by using parallel computing. This implies that the first dimension of a `torch.Tensor` is most often interpreted as the batch dimension and operations are applied *in parallel* to each element of the batch **independently**.
For instance, given a tensor `A` of size `[M,N]` and a tensor `X` of size `[B,N,P]`, the output of `A @ X` will be of size `[B,M,P]`, where the matrix multiplication by `A` is applied to each of the `X[i]` in parallel.

In DiffusionNet, some blocks (*eg* the diffusion block) can be applied to multiple surfaces in parallel so batches are made of surfaces. On the contrary the MLP or gradient features blocks are applied to each vertex independantly so the batches are made of vertices. This will require some particular care when coding.

## 1 - Implementing Diffusion Net

In this section, we seek to implement a version of DiffusionNet which only handles oriented triangle meshes.

It will be impossible for you to test that the output of the functions are the right ones before the complete pipeline is built. However, since shapes are provided in the description of each function, you can ensure each module outputs a tensor of the right shape.

## 1.1 - Some utils

We here code 2 utility functions used in the diffusion block, which allow to project  (resp. unproject) functions to (resp. from) the spectral basis **taking the batch dimension** into account.

For simplicity, we use a batch size = $1$ in this TD as it's tricky to combine functions defined on a mesh with $n_1$ vertices with functions defined on another mesh with $n_2$ vertices.


**Projection**: Recall from TD1, that given functions on a mesh stored in a matrix $\mathbf{f}\in\mathbb{R}^{n\times p}$, the projection into the spectral basis of size $K$ can be done via the matrix multiplication $\Phi^\top A \mathbf{f}$, where $\Phi\in\mathbb{R}^{n\times K}$ are the eigenvectors of the laplacian of the shape and A the area matrix of the shape.

**Reverse projection**: Recall from TD1, that given spectral coefficients of functions on a mesh stored in a matrix $\alpha\in\mathbb{R}^{K\times p}$, the corresponding functions on the mesh are defined as $\Phi \alpha \in\mathbb{R}^{n\times p}$.

### Question 1
**Compute the projection and reverse projection functions, which take batch of functions**

**Tips**:
1. If `A` is to torch Tensor of shape `(B,N,M)`, then `A.mT` does batch-wise matrix transpose and has shape `(B,M,N)`
2. Given a diagonal matrix $A$ and a function $f\in\mathbb{R}^n$, then $Af = \left(A_{ii}f_i\right)$, where $A_{ii}$ are the diagonal values of $A$ (vertex areas in our case). We therefore directly use the diagonal values as input.
3. For matrix multiplication, multiple operations are possibly, namely torch.mm, torch.einsum or the traditional @ operator. Choose what you understand the most easily.


In [ ]:
def project_to_basis(x, evecs, vertex_areas):
    """
    Project an input sinal x to the spectral basis.


    Parameters
    -------------------
    x            : (B, n, p) Tensor of input
    evecs        : (B, n, K) Tensor of eigenvectors
    vertex_areas : (B, n,) vertex areax

    Output
    -------------------
    projected_values : (B, K, p) Tensor of coefficients in the basis
    """
    A = vertex_areas
    Phi = evecs
    Ax = A.unsqueeze(-1) * x # Broadcasting rule!
    projected_values = torch.bmm(Phi.mT, Ax)
    return projected_values ## TODO




def unproject_from_basis(coeffs, evecs):
    """
    Transform input coefficients in basis into a signal on the complete shape.

    Parameters
    -------------------
    coefs : (B, K, p) Tensor of coefficients in the spectral basis
    evecs : (B, n, K) Tensor of eigenvectors

    Output
    -------------------
    decoded_values : (B, n, p) values on each vertex
    """
    Phi = evecs
    decoded_values = torch.bmm(Phi, coeffs)
    return decoded_values ## TODO

## 1.2 - Spectral Diffusion module

The DiffusionModule in Diffusion takes as input a mesh $M$ and set of functions $\left(f_1,\dots, f_p\right)$  on the mesh - that is $f_j:M\to\mathbb{R}$ - and outputs $\left(g_1,\dots, g_p\right)$ with $g_j:M\to\mathbb{R}$ and where $g_j$ the the **diffused version** of $f_j$ after time $t_j$.

Note that each feature function $f_j$ is diffused for time $t_j$ where the time parameter **depends on the index of the function**.
DiffusionNet proposes to **learn each time parameter** $t_1,\dots, t_p$. There is therefore $p$ learnable parameters in this module.


In order to compute Diffusion efficiently, we will use the spectral diffusion approximation introduced in TD1.

Recall from that TD1 that the Laplacian eigenvectors of a mesh made of $N$ vertices can stored in a matrix $\Phi\in\mathbb{R}^{N\times K}$.
Given an initial function $f_0$, its (spectral) diffused version $f_t$ after time $t$ can be written $f_t = \Phi \alpha_t$ with
$$
(\alpha_t)_j = \exp(-t\lambda_j)\beta_j
$$

where $\beta=\Phi^\top A f_0 \ \in\mathbb{R}^K$ is the projection of $f_0$ in the basis, and $A$ the diagonal area matrix.

### Question 2
**Fill the SpectralDiffusion module which diffuses features with learnable time parameters.**

**Clue:**
1. Use [`nn.Parameter`](https://pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html) to create learnable parameters in a module
2. Use [`nn.init.constant_`](https://pytorch.org/docs/stable/nn.init.html#torch.nn.init.constant_) to initialize time parameters to 0
3. Be careful about the dimension of inputs, outputs, intermediate tensors, ...

In [6]:
class SpectralDiffusion(nn.Module):

    def __init__(self, n_channels):
        """
        Initializes the module with time parameters to 0.

        Parameters
        ------------------
        n_channels : int - number of input feature functions
        """
        # This runs the __init__ function of nn.Module
        super().__init__()

        self.n_channels = n_channels

        ## TODO DEFINE AND INITIALIZE THE Diffusion times as learnable parameters.
        shape = (n_channels)
        t = torch.zeros(shape)
        self.diffusion_times = nn.Parameter(t)

    def forward(self, x, evals, evecs, vertex_areas):
        """
        Given input features x and information on the current meshes
        return diffused versions of the features.

        Parameters
        ------------------------
        x     : (B, n, p) batch of input features. p = self.n_channels
        evals : (B, K,) batch of eigenvalues
        evecs : (B, n, K) batch of eigenvectors
        vertex_areas : (B, n,) batch of vertex areax


        Output
        ------------------------
        x_diffuse : diffused version of each input feature
        """
        # Remove negative diffusion times
        with torch.no_grad():
            self.diffusion_times.data = torch.clamp(self.diffusion_times, min=1e-8)

        ## TODO DIFFUSE x
        beta = project_to_basis(x, evecs, vertex_areas) # (B, K, p)
        exp_value = torch.exp(-self.diffusion_times.unsqueeze(0) * evals.unsqueeze(-1))
        alpha = exp_value * beta
        x_diffused = unproject_from_basis(alpha, evecs)
        return x_diffused

## 1.3 Gradient Features module

The gradient feature module allows to compute features from the gradient of the diffused feature. This module treats **each vertex independantly**. The batch size is here the number of vertices and not the shape as a whole.

This modules takes as input **the gradient of each feature** at a vertex, and outputs **a real number for each feature** for this vertex.


The gradient of the features is a **vector field**, so a pair of numbers at each vertex. In practice, the value at a vertex $v$ is stored as a complex number $z\in\mathbb{C}^n$. Because there are $p$ different features for each vertex, we concatenate the $p$ gradients (one for each feature) and obtain a per-vertex vector field $w_v\in\mathbb{C}^{p}$ as **input**.

The gradient feature module therefore takes a vertex embedding $w_v$ as input and outputs a *real-valued* embedding $g_v\in\mathbb{R}^{p}$ where

$$
g_v = \tanh\left(\langle w_v, B w_v\rangle_{\mathbb{C}}\right)
$$
with $\tanh$ and $\langle\rangle_{\mathbb{C}}$ applied **element-wise**.


Where $B$ is a **complex and learnable** matrix, and $\langle\rangle_{\mathbb{C}}$ can be seen as the $\mathbb{R}^2$ inner product when identifying $\mathbb{C}$ with $\mathbb{R}^2$. This means for $a,b,c,d\in\mathbb{R}$ that:
$$
\langle a+ib, c+id\rangle_{\mathbb{C}} = ac + bd
$$


In summary, the gradient feature module:
 - Takes $w_v\in\mathbb{C}^{p}$ as input
 - Computes $g_v$ using the formula above
 - Has $B\in\mathbb{C}^p$ as a unique parameter

### Question 3
**Implement the Gradient Feature Module**

**Tips**:

1. In practice, use $w_v\in\mathbb{R}^{p\times 2}$ and $B=B_{re}+iB_{im}$ where both matrix are real values $p\times p$ matrices. Compute separately the real and imaginary part of $Bw_v$, then apply the inner product and hyperbolic tangent element-wise.
2. A learnable matrix multipication can be represented as a [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) layer

In [ ]:
class SpatialGradient(nn.Module):
    """
    Module which computes g_v from vertex embeddings.
    """
    def __init__(self, n_channels):
        """
        Initializes the module.

        Parameters
        ------------------
        n_channels : int - number of input feature functions
        """

        super().__init__()

        self.n_channels = n_channels

        # Real and Imaginary part of B
        self.B_re = nn.Linear(self.n_channels, self.n_channels, bias=False)
        self.B_im = nn.Linear(self.n_channels, self.n_channels, bias=False)

    def forward(self, vects):
        """
        Parameters
        ----------------------
        Vects : (N, P, 2) per-vertex vector field (w_v)

        Output
        ---------------------
        features : (N, P) per-vertex scalar field
        """
        vects_re = vects[...,0]  # (N,P) real part of w_v
        vects_im = vects[...,1]  # (N,P) imaginary part of w_v

        ## TODO Perform forward pass
        B_vects_re = self.B_re(vects_re) - self.B_im(vects_im)
        B_vects_im = self.B_im(vects_re) + self.B_re(vects_im)
        g_v = torch.tanh(vects_re * B_vects_re + vects_im * B_vects_im)
        return g_v


## 1.4 MLP module

The MLP module is a simple multi-layer perceptron which acts on **each vertex independantly**.

This can be customizable using custom hidden layer sizes and droupout.

### Question 4
**Code the MiniMLP module (with dropout and activation)**

**Tip**:
1. Activation is not applied to the last layer
2. Given a list `layer_list` of `torch.nn` modules, one can generate a large layer using `nn.Sequential(*layer_list)`. See the [documentation](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) for more info
3. Batch is made of per-vertex embedding, not per surface.

In [8]:
class MiniMLP(nn.Sequential):
    '''
    A simple MLP with activation and potential dropout
    '''
    def __init__(self, layer_sizes, dropout=False, activation=nn.ReLU):
        """
        Activation and dropout is applied after all layer BUT the last one

        Parameters
        ---------------------------
        layer_size : list of ints - list of sizes of the MLP
        dropout    : book - whether to add droupout or not
        activation : nn.module : activation function
        """
        super().__init__()

        layer_list = []

        ## TODO FILL THE LAYER LIST
        for i in range(len(layer_sizes)-1):
            layer_list.append(nn.Linear(in_features=layer_sizes[i], out_features=layer_sizes[i+1]))
            if i != len(layer_sizes)-2:
                layer_list.append(activation())
                if dropout:
                    layer_list.append(nn.Dropout())
        self.layer = nn.Sequential(*layer_list)

    def forward(self, x):
        """
        Parameters
        --------------------
        x : (n, p) - input features, batch size is the number of vertices !

        Output
        -------------------
        y : (n, p') - output features
        """
        # NOTHING TO DO HERE
        return self.layer(x)

## 1.5 DiffusionNet Block

![](https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/TD3/DiffusionNet.PNG)

Each diffusion block contains 3 main modules which were implemented in the preceeding three questions. Namely, (1) Learned spectral diffusion, (2) Gradient Feature module and (3) Point-wise features from the MLP module applied sequentially.

Note the following points:
1. Input and output are per-vertex scalars
2. The width (or number of features) is the same as input and output
3. The Diffusion module uses the input features as input.
4. The Spatial Gradient module uses the **gradient** of the output of the Diffusion module as input
5. The MiniMLP uses the **concatenation** of the input features, the output of the Diffusion module and the output of the Spatial Gradient module as input. Its input size is 3 times the number of features.
6. The input features are **added** to the output using a **residual connection**.


### Question 5

**Assemble the Diffusion + Gradient + MLP modules to construct diffusion block using the image and explanation above**

**Tip**
 - We pre-filled the code to compute the gradient of the features
 - We only provide the size of the **hidden** layers of the MLP. You must add the input and output dimension.

In [9]:
class DiffusionNetBlock(nn.Module):
    """
    Complete Diffusion block
    """

    def __init__(self, n_channels, mlp_hidden_dims, dropout=True, use_diffusion=True, use_gradient=True):
        """
        Initializes the module.

        Parameters
        ------------------
        n_channels      : int - number of feature functions (serves as both input and output)
        mlp_hidden_dims : list of int - sizes of HIDDEN layers of the miniMLP.
                          You should add the input and output dimension to it.
        """
        super(DiffusionNetBlock, self).__init__()

        # Specified dimensions
        self.n_channels = n_channels
        self.use_diffusion = use_diffusion
        self.use_gradient = use_gradient
        self.dropout = dropout

        # Diffusion block
        # TODO DEFINE THE 3 SUBPARTS
        layer_sizes = [n_channels + self.use_diffusion * n_channels + self.use_gradient * n_channels] + mlp_hidden_dims + [n_channels]
        if self.use_diffusion:
            self.spectral_diffusion_layer = SpectralDiffusion(n_channels)
        if self.use_gradient:
            self.gradient_layer = SpatialGradient(n_channels)
        self.MiniMLP = MiniMLP(layer_sizes, dropout=dropout)        


    def forward(self, x_in, vertex_areas, evals, evecs, gradX, gradY):
        """
        Parameters
        -------------------
        x_in         : (B,n,p) - Tensor of input signal.
        vertex_areas : (B,n) - Tensor of vertex areas
        evals        : (B, K,) batch of eigenvalues
        evecs        : (B, n, K) batch of eigenvectors
        gradX        : Half of gradient matrix, sparse real tensor with dimension [B,N,N]
        gradY        : Half of gradient matrix, sparse real tensor with dimension [B,N,N]

        Output
        -------------------
        x_out : (B,n,p) - Tensor of output signal.
        """

        # Manage dimensions
        B = x_in.shape[0] # batch dimension

        # Diffusion block
        if self.use_diffusion:
            x_diffuse = self.spectral_diffusion_layer(x_in, evals, evecs, vertex_areas) # DIFFUSED X_in  # (B, N, p)
        else:
            x_diffuse = x_in

        if self.use_gradient:
            # Compute the batch of gradients
            x_grads = [] # Manually loop over the batch
            for b in range(B):
                # gradient after diffusion
                x_gradX = torch.mm(gradX[b,...], x_diffuse[b,...])
                x_gradY = torch.mm(gradY[b,...], x_diffuse[b,...])

                x_grads.append(torch.stack((x_gradX, x_gradY), dim=-1))
            x_grad = torch.stack(x_grads, dim=0)  # (B, N, P, 2)

            # TODO EVALUATE GRADIENT FEATURES
            gradient_features = self.gradient_layer(x_grad)
            # TODO APPLY THE MLP TO THE CONCATENATED FEATURES
            if self.use_diffusion:
                concat = torch.concat([x_in, x_diffuse, gradient_features], dim=2)
            else: 
                concat = torch.concat([x_in, gradient_features], dim=2)
            mlp_output = self.MiniMLP(concat)
            # TODO APPLY THE RESIDUAL CONNECTION
            x_out = mlp_output + x_in
        else:
            # TODO APPLY THE MLP TO THE CONCATENATED FEATURES
            if self.use_diffusion:
                concat = torch.concat([x_in, x_diffuse], dim=2)
            else: 
                concat = torch.concat([x_in], dim=2)
            mlp_output = self.MiniMLP(concat)
            # TODO APPLY THE RESIDUAL CONNECTION
            x_out = mlp_output + x_in

        return x_out

## 1.6 - DiffusionNet

Let's check again the architecture:
![](https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/TD3/DiffusionNet.PNG)

DiffusionNet takes as input per-vertex features of size $p_{in}$. Is first uses a linear layer to transform theses features into $p$-dimensional features, where $p$ is the width of DiffusionNet. There are then multiple blocks of DiffusionNetBlock to produce new features. Similarly to input features, a last linear layer transforms the $p$ dimensional features into vertex features of size $p_{out}$

### Question 6

**Build the DiffusionNet. It consists of an MLP in the first layer to reac with `N_block` of `DiffusionNetBlock` which was implemented in the previous cell.**

In [10]:
class DiffusionNet(nn.Module):

    def __init__(self, p_in, p_out, n_channels=128, N_block=4, last_activation=None, mlp_hidden_dims=None, dropout=True, use_diffusion=True, use_gradient=True):
        """
        Construct a DiffusionNet.
        Parameters
        --------------------
        p_in            : int - input dimension of the network
        p_out           : int - output dimension  of the network
        n_channels      : int - dimension of internal DiffusionNet blocks (default: 128)
        N_block         : int - number of DiffusionNet blocks (default: 4)
        last_activation : int - a function to apply to the final outputs of the network, such as torch.nn.functional.log_softmax
        mlp_hidden_dims : list of int - a list of hidden layer sizes for MLPs (default: [C_width, C_width])
        dropout         : bool - if True, internal MLPs use dropout (default: True)
        """

        super(DiffusionNet, self).__init__()

        ## Store parameters

        # Basic parameters
        self.p_in = p_in
        self.p_out = p_out
        self.n_channels = n_channels
        self.N_block = N_block

        # Outputs
        self.last_activation = last_activation

        # MLP options
        if mlp_hidden_dims == None:
            mlp_hidden_dims = [n_channels, n_channels]
        self.mlp_hidden_dims = mlp_hidden_dims
        self.dropout = dropout


        ## TODO SETUP THE NETWORK (LINEAR LAYERS + BLOCKS)

        self.blocks = [nn.Linear(in_features=p_in, out_features=n_channels)] + [DiffusionNetBlock(n_channels, mlp_hidden_dims, dropout, use_diffusion, use_gradient) for i in range(N_block)] + [nn.Linear(in_features=n_channels, out_features=p_out)] # TOFILL
        self.net = nn.ModuleList(self.blocks) # you can then access each block by self.net[i]


    def forward(self, x_in, vertex_areas, evals=None, evecs=None, gradX=None, gradY=None):
        """
        Progapate a signal through the network.
        Can handle input without batch dimension (will add a dummy dimension to set batch size to 1)

        Parameters
        --------------------
        x_in         : (n,p) or (B,n,p) - Tensor of input signal.
        vertex_areas : (n,) or (B,n) - Tensor of vertex areas
        evals        : (B, K,) or (K,) batch of eigenvalues
        evecs        : (B, n, K) or (n, K) batch of eigenvectors
        gradX        : Half of gradient matrix, sparse real tensor with dimension [N,N] or [B,N,N]
        gradY        : Half of gradient matrix, sparse real tensor with dimension [N,N] or [B,N,N]

        Output
        -----------------------
        x_out (tensor):    Output with dimension [N,C_out] or [B,N,C_out]
        """


        ## Check dimensions, and append batch dimension if not given
        if x_in.shape[-1] != self.p_in:
            raise ValueError(f"DiffusionNet was constructed with p_in={self.p_in}, "
                             f"but x_in has last dim={x_in.shape[-1]}")
        N = x_in.shape[-2]

        if len(x_in.shape) == 2:
            appended_batch_dim = True

            # add a batch dim to all inputs
            x_in = x_in.unsqueeze(0) # (B, N, P)
            vertex_areas = vertex_areas.unsqueeze(0) # (B, N)
            if evals != None: evals = evals.unsqueeze(0) # (B,K)
            if evecs != None: evecs = evecs.unsqueeze(0) # (B,N,K)
            if gradX != None: gradX = gradX.unsqueeze(0) # (B,N,N)
            if gradY != None: gradY = gradY.unsqueeze(0) # (B,N,N)

        elif len(x_in.shape) == 3:
            appended_batch_dim = False

        else: raise ValueError("x_in should be tensor with shape (n,p) or (B,n,p)")

        ##  TODO PROCESS THE INPUTS
        x = (self.net[0])(x_in)
        for i in range(1, len(self.net)-1):
            x_1 = (self.net[i])(x, vertex_areas, evals, evecs, gradX, gradY)
            x = x_1
        if self.last_activation is not None:
            x_final = (self.net[-1])(x)
            x_out = self.last_activation(x_final)
        else:
            x_out = (self.net[-1])(x)
        # Remove batch dim if we added it
        if appended_batch_dim:
            x_out = x_out.squeeze(0) # (N, p_out)

        return x_out

# Part II - RNA segmentation.

**Given meshes of the molecular envelopes for RNA molecules, gathered from the PDB database, we will be segementing them. That is, each vertex is assigned a ground-truth segmentation label according to ~120 functional categories. See [1] for more details. We use this data as a representative task for 3D machine learning on surfaces, predicting the functional segmentation from only the molecule surface shape using DiffusionNet.**


[1] https://hal.inria.fr/hal-02167454v2/document

## 1. Visualize the data

Here is some code to visualize the training data with segmentation of the surfaces.

In [11]:
from rna_dataset import RNAMeshDataset
from mesh_utils.mesh import TriMesh
import matplotlib.pyplot as plt
import plot_utils as plu

In [12]:

root_dir = './RNADataset/'

train_dataset = RNAMeshDataset(root_dir, train=True, num_eig=128, op_cache_dir=None)

loading 60 files: ['5JUO_D.off', '5T62_B.off', '4WCE_Y.off', '4V9K_BB.off', '5E81_16.off', '3J92_7.off', '4V7X_BB.off', '5T7V_C.off', '4TUE_RB.off', '5DGE_3.off', '4V7B_BB.off', '4U4O_3.off', '3JA1_LB.off', '3PIO_Y.off', '5EL7_16.off', '5MDY_3.off', '5H5U_B.off', '4U4O_7.off', '5TBW_AS.off', '5J30_RB.off', '4W4G_RB.off', '4V7J_AB.off', '4V8J_BB.off', '2OTJ_9.off', '4V7X_DB.off', '3J9Y_B.off', '1VQK_9.off', '1Q82_B.off', '4W2H_DB.off', '3G4S_9.off', '1NKW_9.off', '4V5J_BB.off', '3JBN_AB.off', '4V8E_CB.off', '4YZV_YB.off', '4V9P_CB.off', '5J5B_DB.off', '4U6F_7.off', '5OBM_7.off', '3JCD_B.off', '4V55_BA.off', '5IBB_16.off', '1VY7_BB.off', '5GAD_B.off', '4U4Q_3.off', '4WRA_1J.off', '5ADY_A.off', '4V70_BB.off', '4TUB_RB.off', '5AJ0_A4.off', '1KD1_B.off', '4V6N_AA.off', '4V8I_BB.off', '4W4G_YB.off', '4V8F_DB.off', '4V8P_E3.off', '4V5Y_BA.off', '4TUA_YB.off', '5DFE_RB.off', '4V5K_DB.off']


In [13]:
# This loads all operators

data1 = train_dataset[19]
mesh1 = TriMesh(data1['vertices'].numpy(), data1["faces"].numpy())

data2 = train_dataset[37]
mesh2 = TriMesh(data2['vertices'].numpy(), data2["faces"].numpy())


## You can see what kind of information is provided
print(data1.keys())

/content/diffusion_utils.py:196: RuntimeWarning: invalid value encountered in divide
  vertex_normals = vertex_normals / np.linalg.norm(vertex_normals,axis=-1,keepdims=True)
/content/diffusion_utils.py:365: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:654.)
  return torch.sparse.FloatTensor(torch.LongTensor(indices), torch.FloatTensor(values), torch.Size(shape)).coalesce()


dict_keys(['vertices', 'faces', 'frames', 'vertex_area', 'L', 'evals', 'evecs', 'gradX', 'gradY', 'labels', 'labels_og'])


In [14]:

cmap1 = plt.get_cmap("jet")(data1["labels"].numpy() / (train_dataset.n_classes-1))[:,:3]
cmap2 = plt.get_cmap("jet")(data2["labels"].numpy() / (train_dataset.n_classes-1))[:,:3]

# plu.plot(mesh1, cmap1)
plu.plot(mesh2, cmap2)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0466284…

### Load training and validation dataset

All operators and eigenfunctions are precomputed and saved in a cache folder. Computing the cache can take a few minutes.

If you are using colab, you might want first to mount your drive and save the cache there (You can change the `op_cache_dir`) so that you don't have to recompute it everytime. Or either compute the cache once, download it and upload it again everytime. This might be faster than recomputing all along.

To access your google drive, use this code:


```
from google.colab import drive
drive.mount('/content/drive')
```

This makes your Drive available at `/content/drive/My Drive/`

In [15]:
# WARNING: Do not change this cell

from rna_dataset import RNAMeshDataset
from torch.utils.data import DataLoader

root_dir = './RNADataset/'
op_cache_dir = './RNADataset/cache'
num_eig = 128

train_dataset = RNAMeshDataset(root_dir, train=True, num_eig=num_eig, op_cache_dir=op_cache_dir)
train_loader = DataLoader(train_dataset, batch_size=None, shuffle=True, num_workers=0, persistent_workers=False)

valid_dataset = RNAMeshDataset(root_dir, train=False, num_eig=num_eig, op_cache_dir=op_cache_dir)
valid_loader = DataLoader(valid_dataset, batch_size=None, num_workers=0, persistent_workers=False, )

loading 60 files: ['5JUO_D.off', '5T62_B.off', '4WCE_Y.off', '4V9K_BB.off', '5E81_16.off', '3J92_7.off', '4V7X_BB.off', '5T7V_C.off', '4TUE_RB.off', '5DGE_3.off', '4V7B_BB.off', '4U4O_3.off', '3JA1_LB.off', '3PIO_Y.off', '5EL7_16.off', '5MDY_3.off', '5H5U_B.off', '4U4O_7.off', '5TBW_AS.off', '5J30_RB.off', '4W4G_RB.off', '4V7J_AB.off', '4V8J_BB.off', '2OTJ_9.off', '4V7X_DB.off', '3J9Y_B.off', '1VQK_9.off', '1Q82_B.off', '4W2H_DB.off', '3G4S_9.off', '1NKW_9.off', '4V5J_BB.off', '3JBN_AB.off', '4V8E_CB.off', '4YZV_YB.off', '4V9P_CB.off', '5J5B_DB.off', '4U6F_7.off', '5OBM_7.off', '3JCD_B.off', '4V55_BA.off', '5IBB_16.off', '1VY7_BB.off', '5GAD_B.off', '4U4Q_3.off', '4WRA_1J.off', '5ADY_A.off', '4V70_BB.off', '4TUB_RB.off', '5AJ0_A4.off', '1KD1_B.off', '4V6N_AA.off', '4V8I_BB.off', '4W4G_YB.off', '4V8F_DB.off', '4V8P_E3.off', '4V5Y_BA.off', '4TUA_YB.off', '5DFE_RB.off', '4V5K_DB.off']


get_all_operators:   0%|          | 0/60 [00:00<?, ?it/s]

  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- construct

get_all_operators:   0%|          | 0/13 [00:00<?, ?it/s]

  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators
  cache miss -- constructing operators


## 2 -  Training the network

##  <font color='red'>Requirement: we need you to save the model weights and send it with the notebook to be evaluated on this question. Otherwise, this question will not reward points</font>

In order to easily run multiple experiments, we will use a class `Trainer` to run the complete training and testing code.

This way we can simply change parameters as arguments when building the class and run the training easily.

### Question 7: Fill the trainer class

This function is mostly pre-filled. Here are the parts you should implement yourself:

1. In the `__init__` function, **build DiffusionNet** using the `model_cfg` dictionary you will provide as input. An example of model config is the following:

```python
model_cfg = {'inp_feat': 'xyz',  # Type of input Features (xyz, HKS, WKS)
              'num_eig': 32,  # Number of eigenfunctions to use for Spectral Diffusion
              'p_in': 3,  # Number of input features
              'p_out': train_dataset.n_classes,  # Number of output features
              'N_block': 4,  # Number of DiffusionNetBlock
              'n_channels': 128}  # Width of the network
```

2. In the `__init__` function, **define the loss**. Note that segmentation is essentially multi-label classification. The number of classes is given by `train_dataset.n_classes`.

3. Fill in the `forward_step` `train_epoch`, `valid_epoch` methods. Note this is pretty simple and depends on your choice of loss / activation function combination.



In [16]:
# You can remove prints if you want
import numpy as np
from tqdm.notebook import tqdm
class Trainer(object):

    def __init__(self, diffusionnet_cls, model_cfg, train_loader, valid_loader, device='cuda',
                 lr=1e-3, weight_decay=1e-4, num_epochs=200,
                 lr_decay_every = 50, lr_decay_rate = 0.5,
                 log_interval=10, save_dir=None):

        """
        diffusionnet_cls: (nn.Module) class of the DiffusionNet model
        model_cfg: (dict) keyword arguments for model
        train_loader: (torch.utils.DataLoader) DataLoader for training set
        valid_loader: (torch.utils.DataLoader) DataLoader for validation set
        device: (str) 'cuda' or 'cpu'
        lr: (float) learning rate
        weight_decay: (float) weight decay for optimiser
        num_epochs: (int) number of epochs
        lr_decay_every: (int) decay learning rate every this many epochs
        lr_decay_rate: (float) decay learning rate by this factor
        log_interval: (int) print training stats every this many iterations
        save_dir: (str) directory to save model checkpoints
        """

        # TOD build the network from the model_cfg
        p_in = model_cfg['p_in']
        p_out = model_cfg['p_out']
        N_block = model_cfg['N_block']
        n_channels = model_cfg['n_channels']
        use_diffusion = model_cfg.get('use_diffusion', True)
        use_gradient = model_cfg.get('use_gradient', True)
        self.model = diffusionnet_cls(p_in, p_out, n_channels, N_block, use_diffusion=use_diffusion, use_gradient=use_gradient)
        

        self.loss = nn.functional.cross_entropy ## USE A MEANINGFUL LOSS

        ## THIS PART JUST STORES SOME OTHER PARAMETERS
        self.train_loader = train_loader
        self.valid_loader = valid_loader
        self.device = device
        self.lr = lr
        self.weight_decay = weight_decay
        self.num_epochs = num_epochs

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)


        self.lr_decay_every = lr_decay_every
        self.lr_decay_rate = lr_decay_rate
        self.log_interval = log_interval
        self.save_dir = save_dir

        self.train_losses = []
        self.test_losses = []
        self.train_accs = []
        self.test_accs = []

        self.inp_feat = model_cfg.get('inp_feat', 'xyz')
        self.num_eig = model_cfg.get('num_eig', 128)
        if not self.inp_feat in ['xyz', 'hks', 'wks']:
            raise ValueError('inp_feat must be one of xyz, hks, wks')

        self.model.to(self.device)


    def forward_step(self, verts, faces, frames, vertex_area, L, evals, evecs, gradX, gradY):
        """
        Perform a forward step of the model.

        Args:
            verts (torch.Tensor): (N, 3) tensor of vertex positions
            faces (torch.Tensor): (F, 3) tensor of face indices
            frames (torch.Tensor): (N, 3, 3) tensor of tangent frames.
            vertex_area (torch.Tensor): (N, N) sparse Tensor of vertex areas.
            L (torch.Tensor): (N, N) sparse Tensor of cotangent Laplacian.
            evals (torch.Tensor): (num_eig,) tensor of eigenvalues.
            evecs (torch.Tensor): (N, num_eig) tensor of eigenvectors.
            gradX (torch.Tensor): (N, N) tensor of gradient in X direction.
            gradY (torch.Tensor): (N, N) tensor of gradient in Y direction.

        Returns:
            pred (torch.Tensor): (N, p_out) tensor of predicted labels.
        """

        evals = evals[:self.num_eig]
        evecs = evecs[:, :self.num_eig]
        
        if self.inp_feat == 'xyz':
            features = verts
        elif self.inp_feat == 'hks':
            features = self.compute_HKS(evecs, evals, self.num_eig, n_feat=32)
        elif self.inp_feat == 'wks':
            features = self.compute_WKS(evecs, evals, self.num_eig, n_feat=32)
        
        preds = self.model(features, vertex_area, evals=evals, evecs=evecs, gradX=gradX, gradY=gradY)

        # MAYBE ADD ACTIVATION
        return preds



    def train_epoch(self):
        """
        Train the network for one epoch
        """
        train_loss = 0
        train_acc = 0
        for i, batch in enumerate(tqdm(self.train_loader, "Train epoch")):

            verts = batch["vertices"].to(self.device)
            faces = batch["faces"].to(self.device)
            frames = batch["frames"].to(self.device)
            vertex_area = batch["vertex_area"].to(self.device)
            L = batch["L"].to(self.device)
            evals = batch["evals"].to(self.device)
            evecs = batch["evecs"].to(self.device)
            gradX = batch["gradX"].to(self.device)
            gradY = batch["gradY"].to(self.device)
            labels = batch["labels"].to(self.device)

            self.optimizer.zero_grad()

            preds = self.forward_step(verts, faces, frames, vertex_area, L, evals, evecs, gradX, gradY)
            # MAYBE DO SOMETHING TO THE PREDS

            # COMPUTE THE LOSS
            loss = self.loss(preds, labels) #TODO

            loss.backward()
            self.optimizer.step()

            train_loss += loss.item()

            # COMPUTE TRAINING ACCURACY
            pred_labels = torch.max(preds, dim=1).indices # TODO GET PREDICTED LABELS

            n_correct = pred_labels.eq(labels).sum().item() # number of correct predictions
            train_acc += n_correct/labels.shape[0]

        return train_loss/len(self.train_loader), train_acc/len(self.train_loader)

    def valid_epoch(self):
        """
        Run a validation epoch
        """
        val_loss = 0
        val_acc = 0
        print("Start val epoch")
        self.model.eval()                      # set the mode flag — counterpart of .train()
        with torch.no_grad():      
            for i, batch in enumerate(self.valid_loader):

                # READ BATCH
                verts = batch["vertices"].to(self.device)
                faces = batch["faces"].to(self.device)
                frames = batch["frames"].to(self.device)
                vertex_area = batch["vertex_area"].to(self.device)
                L = batch["L"].to(self.device)
                evals = batch["evals"].to(self.device)
                evecs = batch["evecs"].to(self.device)
                gradX = batch["gradX"].to(self.device)
                gradY = batch["gradY"].to(self.device)
                labels = batch["labels"].to(self.device)

                # TODO PERFORM FORWARD STEP
                preds = self.forward_step(verts, faces, frames, vertex_area, L, evals, evecs, gradX, gradY)
                # MAYBE DO SOMETHING TO THE PREDS

                # Compute Loss - THIS DEPENDS ON YOUR CHOICE OF LOSS
                loss = self.loss(preds, labels) ##


                val_loss += loss.item()

                # Compute ACCURACCY
                pred_labels = torch.argmax(preds, dim=1) ## TODO

                n_correct = pred_labels.eq(labels).sum().item() # number of correct predictions
                val_acc += n_correct/labels.shape[0]
        print("End val epoch")
        return val_loss/len(self.valid_loader), val_acc/len(self.valid_loader)

    def run(self):
        os.makedirs('./models', exist_ok=True)
        for epoch in range(self.num_epochs):
            self.model.train()

            if epoch % self.lr_decay_every == 0:
                self.adjust_lr()

            train_ep_loss, train_ep_acc = self.train_epoch()
            self.train_losses.append(train_ep_loss)
            self.train_accs.append(train_ep_acc)

            if epoch % self.log_interval == 0:
                val_loss, val_acc = self.valid_epoch()
                torch.save(self.model.state_dict(), os.path.join(self.save_dir, 'model_latest.pth'))
                print(f'Epoch: {epoch:03d}/{self.num_epochs}, '
                      f'Train Loss: {train_ep_loss:.4f}, '
                      f'Train Acc: {1e2*train_ep_acc:.2f}%, '
                      f'Val Loss: {val_loss:.4f}, '
                      f'Val Acc: {1e2*val_acc:.2f}%')
        torch.save(self.model.state_dict(), os.path.join(self.save_dir, 'model_final.pth'))


    def visualize(self):
        """
        We only test the first two shapes of validation set.
        """
        self.model.eval()
        test_seg_meshes = []

        for i, batch in enumerate(self.valid_loader):
            verts = batch["vertices"].to(self.device)
            faces = batch["faces"].to(self.device)
            frames = batch["frames"].to(self.device)
            vertex_area = batch["vertex_area"].to(self.device)
            L = batch["L"].to(self.device)
            evals = batch["evals"].to(self.device)
            evecs = batch["evecs"].to(self.device)
            gradX = batch["gradX"].to(self.device)
            gradY = batch["gradY"].to(self.device)
            labels = batch["labels"].to(self.device)


            preds = self.forward_step(verts, faces, frames, vertex_area, L, evals, evecs, gradX, gradY)
            pred_labels = torch.max(preds, dim=1).indices

            test_seg_meshes.append([TriMesh(verts.cpu().numpy(), faces.cpu().numpy()),
                                  pred_labels.cpu().numpy()])
            if i==1:
                break


        cmap1 = plt.get_cmap("jet")(test_seg_meshes[0][-1] / (146))[:,:3]
        cmap2 = plt.get_cmap("jet")(test_seg_meshes[1][-1] / (146))[:,:3]

        plu.double_plot(test_seg_meshes[0][0], test_seg_meshes[1][0], cmap1, cmap2)
        #return plot_multi_meshes(test_seg_meshes, cmap='vert_colors')

    def adjust_lr(self):
        lr = self.lr * self.lr_decay_rate
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

    def compute_HKS(self, evecs, evals, num_eig, n_feat):
        """
        Compute the HKS features for each vertex in the mesh.
        Args:
            evecs (torch.Tensor): (N, K) tensor of eigenvectors
            evals (torch.Tensor): (K,) tensor of eigenvectors
            num_eig (int): number of eigenvalues to use
            n_feat (int): number of features to compute

        Returns:
            hks (torch.Tensor): (N, n_feat) tensor of HKS features
        """
        abs_ev = torch.sort(torch.abs(evals)).values[:num_eig]

        # .item() -> Python floats, so NumPy never touches (possibly CUDA) tensors
        t_list = np.geomspace(4*np.log(10)/abs_ev[-1].item(), 4*np.log(10)/abs_ev[1].item(), n_feat)
        t_list = torch.from_numpy(t_list.astype(np.float32)).to(device=evecs.device)

        evals_s = abs_ev

        coefs = torch.exp(-t_list[:,None] * evals_s[None,:])  # (num_T,K)

        natural_HKS = torch.einsum('tk,nk->nt', coefs, evecs[:,:num_eig].square())

        inv_scaling = coefs.sum(1)  # (num_T)

        return (1/inv_scaling)[None,:] * natural_HKS

    def compute_WKS(self, evecs, evals, num_eig, n_feat):
        """
        Compute the WKS features for each vertex in the mesh.

        Args:
            evecs (torch.Tensor): (N, K) tensor of eigenvectors
            evals (torch.Tensor): (K,) tensor of eigenvectors
            num_eig (int): number of eigenvalues to use
            n_feat (int): number of features to compute

        Returns:
            wks: torch.Tensor: (N, num_E) tensor of WKS features
        """
        abs_ev = torch.sort(torch.abs(evals)).values[:num_eig]

        # .item() -> Python floats, so NumPy never touches (possibly CUDA) tensors
        e_min,e_max = np.log(abs_ev[1].item()),np.log(abs_ev[-1].item())
        sigma = 7*(e_max-e_min)/n_feat

        e_min += 2*sigma
        e_max -= 2*sigma

        energy_list = torch.linspace(e_min,e_max,n_feat, device=evecs.device)

        evals_s = abs_ev

        coefs = torch.exp(-torch.square(energy_list[:,None] - torch.log(torch.abs(evals_s))[None,:])/(2*sigma**2))  # (num_E,K)

        natural_WKS = torch.einsum('tk,nk->nt', coefs, evecs[:,:num_eig].square())

        inv_scaling = coefs.sum(1)  # (num_E)
        return (1/inv_scaling)[None,:] * natural_WKS


### Training and visualising segmentation results.

Simply execute the next two cells. There are no TODOs.

In [17]:
model_cfg = {'inp_feat': 'xyz',  # Type of input Features (xyz, HKS, WKS)
              'num_eig': 128,  # Number of eigenfunctions to use for Spectral Diffusion
              'p_in': 3,  # Number of input features
              'p_out': train_dataset.n_classes,  # Number of output features
              'N_block': 4,  # Number of DiffusionNetBlock
              'n_channels': 128 # Width of the network
            }  

# Let's define the trainer
# Save weights directly to Google Drive (persists after the Colab runtime is reset)
save_dir = '/content/drive/MyDrive/Geometry processing and Geometric deep learning/MVA_TD4/models'
os.makedirs(save_dir, exist_ok=True)

my_trainer = Trainer(DiffusionNet, model_cfg, train_loader, valid_loader, save_dir=save_dir)
#my_trainer.run()

# Let's visualize the predicted segmentation at initialization
my_trainer.visualize()

In [18]:
# We can now run the training. IT should take some time (around 2-3 hours). Do not close Colab during the training!
#my_trainer.run()

# And visualize (it's supposed to be clean!)
#my_trainer.visualize()

### Training summary — DiffusionNet on RNA segmentation (xyz input)

**Setup**
- Input features: vertex coordinates (`xyz`, $p_{in}=3$); output: per-vertex logits over `n_classes` segmentation labels
- Architecture: linear layer $3 \to 128$, **4 DiffusionNet blocks** of width 128 (MLP hidden dims $[128, 128]$, dropout 0.5), linear layer $128 \to$ `n_classes`
- Spectral basis: 128 eigenpairs of the Laplacian (`num_eig = 128`)
- Loss: cross-entropy on raw logits (no final activation, since `cross_entropy` applies `log_softmax` internally)
- Optimiser: Adam, weight decay $10^{-4}$, 200 epochs on the 60 training meshes (one mesh per step), on a GPU (T4)
- Validation runs in `eval()` mode under `torch.no_grad()`, so dropout is disabled

**Results**

| Epoch | Train loss | Train acc | Val loss | Val acc |
|------:|-----------:|----------:|---------:|--------:|
| 0     | 4.84 | 1.6 %  | 4.63 | 4.1 %  |
| 10    | 1.90 | 37.3 % | 1.51 | 50.0 % |
| 50    | 0.94 | 66.1 % | 0.75 | 72.2 % |
| 100   | 0.75 | 72.6 % | 0.63 | 76.6 % |
| 150   | 0.66 | 75.8 % | 0.57 | 79.2 % |
| 190   | 0.61 | 77.5 % | 0.57 | **79.5 %** |

**Analysis**
- The initial loss (4.84) is close to $\ln(\text{n\_classes})$, which is what an uninformed classifier should give. This confirms that the logits and the loss are set up consistently.
- Both losses decrease smoothly and validation accuracy rises from ~4 % to **~80 %**, against roughly 1 % for random guessing. Most of the gain happens in the first 100 epochs; validation accuracy plateaus around 79–80 % after epoch ~140.
- **No overfitting** is visible: the validation loss keeps decreasing until the end.
- Validation accuracy is *higher* than training accuracy throughout. This is expected: training accuracy is measured with dropout active and averaged over an epoch while the weights are still changing, whereas validation uses the full deterministic network at the end of the epoch.
- The predicted segmentations on the two validation meshes are visually coherent: labels form contiguous regions that follow the shape of the molecular surface.
- Note: the provided `adjust_lr` always sets the learning rate to `lr * lr_decay_rate` from the *initial* value, so the learning rate stays constant at $5 \cdot 10^{-4}$ for the whole run instead of decaying every 50 epochs. A real step decay might help push past the plateau.

**Weights:** `model_final.pth` (submitted with this notebook).

This run is the **baseline** for the ablations in Question 8.


### Question 8: Ablation studies

#### This is the final question of this lab which involves several subquestions. The objective is to understand the utility of different components of DiffusionNet block built during this session. More specifically, we will make one change at a time to the above network to understand how the segmentation accuracy varies.


1. Remove the learned diffusion module such that a "Diffusion" block consists of MLP + Gradient features.

2. Remove the Gradient feature module such that a "Diffusion" block consists of Diffusion + MLP features.

3. Instead of feeding in the `xyz` coordinate as features, what happens if either of HKS and/or WKS is used as a feature?


The first two question consists of reimplementing `DiffusionNet` module such that it incorporates aforementioned changes into the blocks. Either create a new class or add arguments which you'll use in the `model_cfg` dictionary.

For the last question, please pay attention to `inp_feat` and `p_in`.

In [19]:
### TODO:
# 1. Ablation of the diffusion module

model_cfg = {'inp_feat': 'xyz',  # Type of input Features (xyz, HKS, WKS)
            'num_eig': 128,  # Number of eigenfunctions to use for Spectral Diffusion
            'p_in': 3,  # Number of input features
            'p_out': train_dataset.n_classes,  # Number of output features
            'N_block': 4,  # Number of DiffusionNetBlock
            'n_channels': 128, # Width of the network
            'use_diffusion': False,
            'use_gradient': True
            }  

# Let's define the trainer
# Save weights directly to Google Drive (persists after the Colab runtime is reset)
save_dir = '/content/drive/MyDrive/Geometry processing and Geometric deep learning/MVA_TD4/models_diffusion_ablation'
os.makedirs(save_dir, exist_ok=True)

my_trainer = Trainer(DiffusionNet, model_cfg, train_loader, valid_loader, save_dir=save_dir)

# Let's visualize the predicted segmentation at initialization
my_trainer.visualize()


In [20]:
# Ablation study of diffusion: training
# my_trainer.run()
# my_trainer.visualize()

### Ablation 1 — Removing the learned diffusion (MLP + gradient features)

**Setup:** identical to the baseline (xyz input, 4 blocks of width 128, 128 eigenpairs, cross-entropy, 200 epochs), except that each block uses `use_diffusion=False`:
- no `SpectralDiffusion` module;
- the gradient features are computed directly from the block input $x_{in}$ instead of the diffused signal;
- the MLP therefore receives $[x_{in},\ \text{grad features}]$, so its input width is $2p = 256$ instead of $3p = 384$.

**Results (vs. baseline)**

| Epoch | Train loss | Train acc | Val loss | Val acc | Baseline val acc |
|------:|-----------:|----------:|---------:|--------:|-----------------:|
| 0   | 4.89 | 1.2 %  | 4.78 | 2.2 %  | 4.1 %  |
| 10  | 3.83 | 6.4 %  | 3.55 | 10.6 % | 50.0 % |
| 20  | 3.52 | 8.9 %  | 3.24 | 13.3 % | 63.3 % |
| 40  | 3.29 | 11.9 % | 3.00 | 18.2 % | 70.5 % |
| 60  | 3.14 | 14.4 % | 2.78 | 25.1 % | 74.2 % |
| 80  | 3.03 | 16.4 % | 2.66 | 26.5 % | 75.9 % |
| 100 | 2.95 | 17.8 % | 2.58 | 28.5 % | 76.6 % |
| 120 | 2.89 | 18.9 % | 2.57 | 28.1 % | 77.7 % |
| 140 | 2.86 | 19.3 % | 2.49 | 28.7 % | 79.1 % |
| 160 | 2.81 | 20.3 % | 2.42 | 31.6 % | 78.9 % |
| 180 | 2.79 | 20.9 % | 2.41 | 30.8 % | 79.9 % |
| 190 | 2.76 | 21.5 % | 2.36 | **34.0 %** | **79.5 %** |

| Final (epoch 190) | Val loss | Val acc |
|---|---:|---:|
| Baseline (diffusion + gradient) | 0.57 | 79.5 % |
| **No diffusion** | 2.36 | 34.0 % |
| Difference | +1.79 | **−45.5 pts** |

**Analysis**
- Removing diffusion causes a **large drop in performance**: final validation accuracy is about **34 %**, against **79.5 %** for the baseline, a gap of about 45 points. The validation loss is about 4× higher.
- The ablated model learns **much more slowly**. The baseline reaches 50 % validation accuracy within 10 epochs; without diffusion, the model never gets there in 200 epochs.
- It has **not fully converged**: the training and validation losses are still decreasing at epoch 190, and validation accuracy keeps climbing slowly, though noisily (28–34 % over the last 60 epochs). Longer training would probably improve it somewhat, but at this pace it would not come close to the baseline.
- It remains well above chance (about 1 %), so the xyz coordinates and the local gradient features do carry some information.
- **Interpretation: receptive field.** The gradient operators `gradX`/`gradY` are sparse and only involve a vertex's immediate neighbours, so each block propagates information by about one ring. With 4 blocks, each vertex only "sees" a small neighbourhood. Learned diffusion, in contrast, can propagate information over an arbitrarily large region in a single block, with its extent controlled by the learned time $t$. The results are consistent with segmentation of functional regions requiring this larger-scale context, making diffusion the **core component** of DiffusionNet.


In [21]:
### TODO:
# 2. Ablation of the gradient features

model_cfg = {'inp_feat': 'xyz',  # Type of input Features (xyz, HKS, WKS)
            'num_eig': 128,  # Number of eigenfunctions to use for Spectral Diffusion
            'p_in': 3,  # Number of input features
            'p_out': train_dataset.n_classes,  # Number of output features
            'N_block': 4,  # Number of DiffusionNetBlock
            'n_channels': 128, # Width of the network
            'use_diffusion': True,
            'use_gradient': False
            }  

# Let's define the trainer
# Save weights directly to Google Drive (persists after the Colab runtime is reset)
save_dir = '/content/drive/MyDrive/Geometry processing and Geometric deep learning/MVA_TD4/models_gradient_ablation'
os.makedirs(save_dir, exist_ok=True)

my_trainer = Trainer(DiffusionNet, model_cfg, train_loader, valid_loader, save_dir=save_dir)

# Let's visualize the predicted segmentation at initialization
my_trainer.visualize()


In [22]:
# Ablation study of gradient: training
# my_trainer.run()
my_trainer.visualize()

### Ablation 2 — Removing the gradient features (Diffusion + MLP)

**Setup:** identical to the baseline (xyz input, 4 blocks of width 128, 128 eigenpairs, cross-entropy, 200 epochs), except that each block uses `use_gradient=False`:
- no `SpatialGradient` module, and the gradients `gradX`/`gradY` are not computed;
- the MLP therefore receives $[x_{in},\ x_{diffuse}]$, so its input width is $2p = 256$ instead of $3p = 384$.

**Results (vs. baseline)**

| Epoch | Train loss | Train acc | Val loss | Val acc | Baseline val acc |
|------:|-----------:|----------:|---------:|--------:|-----------------:|
| 0   | 4.83 | 1.5 %  | 4.66 | 2.7 %  | 4.1 %  |
| 10  | 2.83 | 17.2 % | 2.36 | 27.1 % | 50.0 % |
| 20  | 2.32 | 25.8 % | 2.10 | 29.7 % | 63.3 % |
| 40  | 2.04 | 31.6 % | 1.77 | 38.9 % | 70.5 % |
| 60  | 1.84 | 36.4 % | 1.55 | 45.1 % | 74.2 % |
| 80  | 1.70 | 40.2 % | 1.43 | 49.5 % | 75.9 % |
| 100 | 1.57 | 43.8 % | 1.31 | 52.2 % | 76.6 % |
| 120 | 1.46 | 47.2 % | 1.23 | 54.8 % | 77.7 % |
| 140 | 1.43 | 48.0 % | 1.15 | 59.0 % | 79.1 % |
| 160 | 1.32 | 51.8 % | 1.04 | 62.1 % | 78.9 % |
| 180 | 1.27 | 53.2 % | 1.00 | 62.9 % | 79.9 % |
| 190 | 1.26 | 53.6 % | 1.01 | **63.4 %** | **79.5 %** |

| Final (epoch 190) | Val loss | Val acc |
|---|---:|---:|
| Baseline (diffusion + gradient) | 0.57 | 79.5 % |
| **No gradient features** | 1.01 | 63.4 % |
| Difference | +0.44 | **−16.1 pts** |

**Analysis**
- Removing the gradient features causes a **clear but moderate drop**: final validation accuracy is about **63 %**, against **79.5 %** for the baseline, a gap of about 16 points.
- Learning is **much slower**: the model reaches 50 % validation accuracy only around epoch 90, while the baseline gets there within 10 epochs.
- It has **not converged**: both losses are still decreasing at epoch 190, and validation accuracy is still rising (about 60 → 63 % over the last 40 epochs). Longer training would likely narrow the gap somewhat, so the 16-point difference partly reflects slower convergence, not only a lower final capacity.
- **Interpretation: isotropy.** Spectral diffusion acts on each frequency with the factor $e^{-\lambda t}$, which spreads information **equally in all directions** around a vertex (it is isotropic). The gradient features, with their learnable complex matrix $B$, are the only component that captures **directional** information, such as how features vary relative to each other along the surface. Without them, the network can still gather context from a large neighbourhood (thanks to diffusion), but it cannot distinguish directions within it. The results are consistent with this directional information mattering for fine segmentation boundaries, though it is less critical than diffusion.

### Comparison of the two ablations

| Model | Final val acc | Gap to baseline |
|---|---:|---:|
| Baseline (diffusion + gradient) | 79.5 % | — |
| No gradient features | 63.4 % | −16.1 pts |
| No diffusion | 34.0 % | −45.5 pts |

Removing diffusion hurts **about three times more** than removing the gradient features. Diffusion provides the large receptive field, which is the essential ingredient; the gradient features add directional information on top of it, which improves accuracy and convergence speed significantly. Both components are useful, and the full DiffusionNet block clearly outperforms either ablation.


In [23]:
# 3. Training with HKS instead of xyz
 
model_cfg = {'inp_feat': 'hks',  # Type of input Features (xyz, HKS, WKS)
            'num_eig': 128,  # Number of eigenfunctions to use for Spectral Diffusion
            'p_in': 32,  # Number of input features
            'p_out': train_dataset.n_classes,  # Number of output features
            'N_block': 4,  # Number of DiffusionNetBlock
            'n_channels': 128, # Width of the network
            'use_diffusion': True,
            'use_gradient': True
            }  

# Let's define the trainer
# Save weights directly to Google Drive (persists after the Colab runtime is reset)
save_dir = '/content/drive/MyDrive/Geometry processing and Geometric deep learning/MVA_TD4/models_HKS'
os.makedirs(save_dir, exist_ok=True)

my_trainer = Trainer(DiffusionNet, model_cfg, train_loader, valid_loader, save_dir=save_dir)

# Let's visualize the predicted segmentation at initialization
my_trainer.visualize()


In [24]:
my_trainer.run()
my_trainer.visualize()

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Start val epoch
End val epoch
Epoch: 000/200, Train Loss: 4.7806, Train Acc: 2.40%, Val Loss: 4.2848, Val Acc: 5.05%


Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

...

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

### Input features — HKS instead of xyz

**Setup:** identical to the baseline (full DiffusionNet: diffusion + gradient features, 4 blocks of width 128, 128 eigenpairs, cross-entropy, 200 epochs), except for the input features:
- `inp_feat = 'hks'`: each vertex is described by its **Heat Kernel Signature** at 32 time scales, $HKS(x, t) = \sum_k e^{-t\lambda_k}\,\phi_k(x)^2$ (normalised per time), with the times spaced geometrically between $4\ln 10/\lambda_{max}$ and $4\ln 10/\lambda_1$;
- hence $p_{in} = 32$ instead of 3.

**Results (vs. xyz baseline)**

| Epoch | Train loss | Train acc | Val loss | Val acc | Baseline (xyz) val acc |
|------:|-----------:|----------:|---------:|--------:|-----------------------:|
| 0   | 4.78 | 2.4 %  | 4.28 | 5.1 %  | 4.1 %  |
| 10  | 1.32 | 56.1 % | 1.00 | 66.1 % | 50.0 % |
| 20  | 1.01 | 65.2 % | 0.82 | 71.5 % | 63.3 % |
| 40  | 0.82 | 71.5 % | 0.72 | 74.9 % | 70.5 % |
| 60  | 0.74 | 74.2 % | 0.70 | 76.0 % | 74.2 % |
| 80  | 0.69 | 75.8 % | 0.66 | 77.4 % | 75.9 % |
| 100 | 0.64 | 77.5 % | 0.62 | 78.7 % | 76.6 % |
| 120 | 0.68 | 76.5 % | 0.61 | 79.2 % | 77.7 % |
| 140 | 0.61 | 78.8 % | 0.61 | 79.6 % | 79.1 % |
| 160 | 0.59 | 79.6 % | 0.60 | 79.9 % | 78.9 % |
| 180 | 0.58 | 80.0 % | 0.59 | 80.6 % | 79.9 % |
| 190 | 0.56 | 80.6 % | 0.58 | **81.0 %** | **79.5 %** |

| Final (epoch 190) | Train acc | Val loss | Val acc |
|---|---:|---:|---:|
| Baseline (xyz) | 77.5 % | 0.57 | 79.5 % |
| **HKS** | 80.6 % | 0.58 | **81.0 %** |
| Difference | +3.1 pts | +0.01 | **+1.5 pts** |

**Analysis**
- HKS input gives a **slightly better final accuracy** (81.0 % vs 79.5 %) with essentially the same validation loss. With a single training run per configuration, a 1.5-point difference is within the range of run-to-run variability, so the final performance should be considered **comparable to slightly better**.
- The clearest effect is **much faster convergence**: HKS reaches 66 % validation accuracy after 10 epochs (vs 50 % for xyz) and 75 % after 40 epochs, a level that xyz only reaches around epoch 70.
- **Interpretation.** HKS is an **intrinsic** descriptor: it depends only on the Laplace–Beltrami operator, so it is invariant to rigid motions (rotations and translations) of the molecule and robust to near-isometric deformations. It is also **multi-scale**: small $t$ captures local curvature-like geometry, large $t$ captures the global position of a vertex on the surface. xyz coordinates, in contrast, depend on how each molecule is positioned and oriented in space, and the network must learn this information from scratch. With HKS, part of the geometric context is already encoded in the input, which explains the head start. After enough training, DiffusionNet can extract comparable information from raw xyz through its diffusion and gradient layers, so the gap mostly closes.


In [25]:
# 3. Training with WKS instead of xyz
 
model_cfg = {'inp_feat': 'wks',  # Type of input Features (xyz, HKS, WKS)
            'num_eig': 128,  # Number of eigenfunctions to use for Spectral Diffusion
            'p_in': 32,  # Number of input features
            'p_out': train_dataset.n_classes,  # Number of output features
            'N_block': 4,  # Number of DiffusionNetBlock
            'n_channels': 128, # Width of the network
            'use_diffusion': True,
            'use_gradient': True
            }  

# Let's define the trainer
# Save weights directly to Google Drive (persists after the Colab runtime is reset)
save_dir = '/content/drive/MyDrive/Geometry processing and Geometric deep learning/MVA_TD4/models_WKS'
os.makedirs(save_dir, exist_ok=True)

my_trainer = Trainer(DiffusionNet, model_cfg, train_loader, valid_loader, save_dir=save_dir)

# Let's visualize the predicted segmentation at initialization
my_trainer.visualize()


In [26]:
my_trainer.run()
my_trainer.visualize()

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Start val epoch
End val epoch
Epoch: 000/200, Train Loss: 4.8153, Train Acc: 1.78%, Val Loss: 4.6284, Val Acc: 2.73%


Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

...

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

Train epoch:   0%|          | 0/60 [00:00<?, ?it/s]

### Input features — WKS instead of xyz

**Setup:** identical to the baseline (full DiffusionNet: diffusion + gradient features, 4 blocks of width 128, 128 eigenpairs, cross-entropy, 200 epochs), except for the input features:
- `inp_feat = 'wks'`: each vertex is described by its **Wave Kernel Signature** at 32 energy levels, $WKS(x, e) = \sum_k \exp\!\left(-\frac{(e - \log \lambda_k)^2}{2\sigma^2}\right)\phi_k(x)^2$ (normalised per energy), with the energies spaced evenly in $\log \lambda$;
- hence $p_{in} = 32$ instead of 3.

**Results (vs. xyz baseline and HKS)**

| Epoch | Train loss | Train acc | Val loss | Val acc | xyz val acc | HKS val acc |
|------:|-----------:|----------:|---------:|--------:|------------:|------------:|
| 0   | 4.82 | 1.8 %  | 4.63 | 2.7 %  | 4.1 %  | 5.1 %  |
| 10  | 1.62 | 48.9 % | 1.26 | 59.1 % | 50.0 % | 66.1 % |
| 20  | 1.11 | 62.1 % | 0.92 | 68.4 % | 63.3 % | 71.5 % |
| 40  | 0.88 | 69.5 % | 0.81 | 72.6 % | 70.5 % | 74.9 % |
| 60  | 0.77 | 72.9 % | 0.74 | 75.2 % | 74.2 % | 76.0 % |
| 80  | 0.72 | 75.0 % | 0.76 | 75.0 % | 75.9 % | 77.4 % |
| 100 | 0.67 | 76.5 % | 0.72 | 76.2 % | 76.6 % | 78.7 % |
| 120 | 0.64 | 77.8 % | 0.72 | 76.2 % | 77.7 % | 79.2 % |
| 140 | 0.63 | 78.1 % | 0.70 | 77.4 % | 79.1 % | 79.6 % |
| 160 | 0.61 | 79.1 % | 0.69 | 78.0 % | 78.9 % | 79.9 % |
| 180 | 0.59 | 79.7 % | 0.68 | 78.1 % | 79.9 % | 80.6 % |
| 190 | 0.59 | 79.7 % | 0.68 | **78.5 %** | **79.5 %** | **81.0 %** |

| Final (epoch 190) | Train acc | Val loss | Val acc |
|---|---:|---:|---:|
| xyz | 77.5 % | 0.57 | 79.5 % |
| HKS | 80.6 % | 0.58 | 81.0 % |
| **WKS** | 79.7 % | 0.68 | **78.5 %** |

**Analysis**
- WKS reaches a **final accuracy comparable to xyz** (78.5 % vs 79.5 %) and slightly below HKS (81.0 %). With a single run per configuration, these differences of 1–2 points are within run-to-run variability.
- Like HKS, WKS gives a **faster start** than xyz (59 % after 10 epochs vs 50 %), but a slower one than HKS (66 %).
- **Generalisation gap:** WKS is the only configuration where training accuracy ends **above** validation accuracy (79.7 % vs 78.5 %), and its validation loss stops improving from epoch ~60 (about 0.68–0.76) while the training loss keeps decreasing. Its final validation loss (0.68) is clearly higher than with xyz or HKS (0.57–0.58). This suggests **mild overfitting** with WKS features.
- **Interpretation.** Both descriptors are intrinsic and multi-scale, but they filter the spectrum differently. HKS is a **low-pass** filter ($e^{-t\lambda}$): at large $t$ it is dominated by the low frequencies, which encode the global position of a vertex on the surface. WKS is a **band-pass** filter in $\log\lambda$: each feature isolates a frequency band, and it was designed to be highly discriminative of fine local geometry while giving little weight to the lowest frequencies. Segmenting large functional regions relies heavily on coarse, global context, which HKS provides more directly. WKS's fine-scale detail is more specific to each training shape, which is consistent with the larger generalisation gap.




### Summary of Question 8

| Configuration | Input | Diffusion | Gradient | Final val acc | Δ vs baseline | Val acc @ epoch 10 |
|---|---|:---:|:---:|---:|---:|---:|
| **Baseline** | xyz | ✓ | ✓ | 79.5 % | — | 50.0 % |
| No diffusion | xyz | ✗ | ✓ | 34.0 % | −45.5 pts | 10.6 % |
| No gradient features | xyz | ✓ | ✗ | 63.4 % | −16.1 pts | 27.1 % |
| HKS input | HKS (32) | ✓ | ✓ | 81.0 % | +1.5 pts | 66.1 % |
| WKS input | WKS (32) | ✓ | ✓ | 78.5 % | −1.0 pts | 59.1 % |

**Conclusions**
**Learned diffusion is the core of DiffusionNet.** Removing it divides accuracy by more than two, because without it each block only propagates information about one ring of neighbours.
**Gradient features matter, but less.** They add directional (anisotropic) information on top of the isotropic diffusion. Removing them costs 16 points and slows convergence considerably.
Finally, **The choice of input features matters much less than the architecture.** Intrinsic descriptors (HKS, WKS) mainly speed up early training. Final accuracies stay within about 2 points of xyz, as DiffusionNet learns comparable geometric information from raw coordinates. HKS gives the best result overall; WKS shows signs of overfitting.
In the end the three xyz, HKS and WKS don't show significant differences in terms of performance given this configuration of training. 
It may be intersting to perform those ablation studies for WKS and HKS to evaluate the robustess of these setup.

Disclosure: My favorite LLM helped me notice some issues and debugging, namely for the HKS and WKS training as well as construct the comparatives tables. 